# Install and import libraries

In [109]:
import sys
print(sys.executable)

/Users/ioana/.pyenv/versions/tf-env/bin/python


In [110]:
%pip install pydot
%pip install tensorflow
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [111]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from pathlib import Path

from constants import (
    DATA_INPUT_PATH,
    MODEL_PATH,
    METADATA_PATH,
)

CLASSES = {
    0: "resting",
    1: "palm up",
    2: "closed fist",
    3: "ok",
    4: "pointer finger",
    5: "peace",
    6: "shaa",
    7: "peace among worlds"
}

# Read the files in the data dir

In [112]:
# Read all of the files in the data folder
files_in_folder = Path(DATA_INPUT_PATH).glob("*.csv")

files = [x for x in files_in_folder]
print([file for file in files])

[PosixPath('data/emg_gestures_data_20250205_184712.csv'), PosixPath('data/emg_gestures_data_20250205_212859.csv'), PosixPath('data/emg_gestures_data_20250201_215436.csv'), PosixPath('data/emg_gestures_data_20250128_220435.csv'), PosixPath('data/emg_gestures_data_20250226_122030.csv'), PosixPath('data/emg_gestures_data_20250202_172502.csv'), PosixPath('data/emg_gestures_data_20250219_122322.csv')]


## Convert .csv(s) to dataframes and concatenate

In [113]:
# Read the data from the files
dfs = []

for file in files:
    df = pd.read_csv(str(file))
    dfs.append(df)
        
# Convert the data to a DataFrame
df = pd.concat([x for x in dfs], axis=0)

columns = ["gesture_id", "s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8"]
df = df[columns]

print(df.head())

# Before removing duplicates
print(f"Shape of dataframe before removing duplicates {df.shape}")
# Remove duplicates
df = df.drop_duplicates()
print(f"Shape of dataframe after removing duplicates {df.shape}")
print(df)

   gesture_id   s1   s2   s3   s4   s5   s6  s7  s8
0           0  474  264  409  136  122  159  92  85
1           0  238  188  506   86   67   91  50  43
2           0  133  208  505   85   54   82  43  39
3           0  114  194  500   80   52   73  37  35
4           0   94  208  535   84   55   61  36  32
Shape of dataframe before removing duplicates (117348, 9)
Shape of dataframe after removing duplicates (117348, 9)
      gesture_id   s1   s2   s3   s4   s5   s6   s7   s8
0              0  474  264  409  136  122  159   92   85
1              0  238  188  506   86   67   91   50   43
2              0  133  208  505   85   54   82   43   39
3              0  114  194  500   80   52   73   37   35
4              0   94  208  535   84   55   61   36   32
...          ...  ...  ...  ...  ...  ...  ...  ...  ...
9825           7   40  128   90   45   68   68  257  124
9826           7   46  161  163   56   86   67  270  136
9827           7   45  168  170   56   95   62  278  138
982

In [116]:
import numpy as np
import pandas as pd

# Parameters
window_size = 10  # Number of past samples to include
max_value = 255  # Assuming 8-bit sensor data

# Extract feature columns (excluding label, if it exists)
feature_columns = [col for col in df.columns if col != 'Label']
features = df[feature_columns].to_numpy()

# Convert features into a 3D array: [samples, features, time]
n_samples = len(features)
n_features = len(feature_columns)
reshaped_data = []

for i in range(n_samples):
    # Get the last `window_size` rows or fewer for each sample
    window = features[max(i - window_size + 1, 0):i + 1]
    
    # Pad the window if it has fewer than `window_size` rows
    if len(window) < window_size:
        padding = np.zeros((window_size - len(window), n_features))
        window = np.vstack((padding, window))
    
    reshaped_data.append(window)

# Convert list to 3D numpy array
reshaped_data = np.stack(reshaped_data, axis=0)
reshaped_data = reshaped_data.transpose(0, 2, 1)  # Shape: [samples, features, time]

# Compute mean and std per feature across time and samples
mean = np.mean(reshaped_data[:, 1:9, :], axis=2, keepdims=True)  # Shape: (samples, features-1, 1)
std = np.std(reshaped_data[:, 1:9, :], axis=2, keepdims=True)    # Shape: (samples, features-1, 1)

# Avoid division by zero by replacing zeros in std with 1
std = np.where(std == 0, 1, std)

# Compute RMS features ONLY on columns 1 to 8
scaled_data = np.sqrt(np.mean(np.square(reshaped_data[:, 1:9, :]), axis=2))  # Shape: (samples, features-1)
normalized_data = (scaled_data - mean.squeeze()) / std.squeeze() * max_value  # Normalize only features 1-8

# Reconstruct the final dataframe
df_transformed = df.copy()
df_transformed.iloc[:, 1:8] = normalized_data.round().astype(int)  # Apply normalization only to columns 1-8

# Print final DataFrame
print(df)
df.iloc[:, 1:8] = df_transformed[1:8]
print(df)


ValueError: setting an array element with a sequence.

## Scale and clean the data, then train the model

In [ ]:
X = df.drop(columns=['gesture_id'])
y = df['gesture_id']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape the data for Conv1D
X_train_scaled = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_scaled = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# # filter for valid classes because the data is not clean
# valid_classes = [0, 2, 3]
# mask_train = y_train.isin(valid_classes)
# mask_test = y_test.isin(valid_classes)

# # apply the mask to the training and testing data
# X_train_filtered = X_train_scaled[mask_train]
# y_train_filtered = y_train[mask_train]

# X_test_filtered = X_test_scaled[mask_test]
# y_test_filtered = y_test[mask_test]

# one hot encode the target data
y_train_categorical = to_categorical(y_train, num_classes=8)
y_test_categorical = to_categorical(y_test, num_classes=8)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(8, activation='softmax')
])

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Compile the model with categorical crossentropy for multi-class classification
model.compile(
    optimizer=Adam(learning_rate=0.001), 
    loss='categorical_crossentropy', 
    metrics=['accuracy'],
)

# Train the model using the filtered training data
model.fit(
    X_train_scaled, 
    y_train_categorical, 
    epochs=100, 
    batch_size=64, 
    validation_split=0.2, 
    callbacks=[early_stopping]
)

# Evaluate the model on the filtered test set
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_categorical)

print(f"Test accuracy: {test_acc}")

In [ ]:
# Save the model
model.save(f"model/{MODEL_PATH}")

# Save the scaler and the column names to a pickle file
with open(f"model/{METADATA_PATH}", 'wb') as f:
    pickle.dump((scaler, X_train.columns), f)

In [ ]:
# thumbs-up then peace-sign
emg_data = [
    [160,245,126,32,28,25,26,99], # thumbs-up
    [67,197,559,104,41,82,257,169], # peace-sign
    [205,440,165,40,27,83,229,226], # gun-fingers
    [416,434,134,71,36,48,102,103] # fist
]

for data in emg_data:
    emg_features_df = pd.DataFrame([data], columns=X_train.columns)

    emg_features_scaled = scaler.transform(emg_features_df)
    emg_features_reshaped = emg_features_scaled.reshape(1, -1)

    prediction = model.predict(emg_features_reshaped)
    predicted_class = np.argmax(prediction)

    print(f"Predicted class: {predicted_class} - {CLASSES[predicted_class]}")
